# 01 · Data discovery — NHS RTT monthly extract

**Purpose:** understand the *structure* of a raw monthly NHS RTT CSV before any
cleaning, transformation, or interpretation.

**Development month:** June 2026 (`data/raw/rtt_2026_06.csv`) — the single file
in scope for Phase 1 (see `docs/decisions.md` D-001).

All profiling logic lives in `src/nhs_rtt/profile.py`. This notebook only
*calls* it and renders the results — it deliberately contains no profiling
logic of its own.

Rules honoured here:

- Raw files are read-only; nothing is written back to `data/raw/`.
- No rows/columns removed, no types coerced, blanks **counted, not interpreted**.
- Large per-row tables are written to `outputs/profiles/`; the compact
  overview is rendered inline for convenience.
- The *Observations* section at the end is hand-written narrative, not a
  generated assertion.

In [1]:
import sys
from pathlib import Path

# Make ``src/`` importable when running the notebook from ``notebooks/`` or the
# repo root. (The package is also installed editable into the approved venv.)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from nhs_rtt import profile as rtt_profile

RAW_CSV = ROOT / "data" / "raw" / "rtt_2026_06.csv"
PROFILE_DIR = ROOT / "outputs" / "profiles"
RAW_CSV, RAW_CSV.exists()

(WindowsPath('C:/Users/abdul/OneDrive/Desktop/P_Projects/NHS-RTT-Waiting-Times/data/raw/rtt_2026_06.csv'),
 True)

In [2]:
# Single streaming pass; also writes CSV + Markdown into outputs/profiles/.
prof, written = rtt_profile.profile_and_save(RAW_CSV, outdir=PROFILE_DIR)

print(f"{Path(prof.path).name}: {prof.n_rows:,} rows x {prof.n_cols} cols")
print(f"duplicate rows (exact) : {prof.duplicate_row_count:,}")
print(f"ragged rows            : {prof.n_ragged_rows:,}")
print(f"file on disk           : {prof.file_bytes:,} bytes")
print(f"est. in-memory         : ~{prof.est_memory_bytes:,} bytes (estimate only)")
print(f"structurally clean     : {prof.is_structurally_clean}")
print("\nfiles written:")
for w in written:
    print(" ", w.relative_to(ROOT))

rtt_2026_06.csv: 182,411 rows x 121 cols
duplicate rows (exact) : 0
ragged rows            : 0
file on disk           : 82,084,725 bytes
est. in-memory         : ~836,517,419 bytes (estimate only)
structurally clean     : True

files written:
  outputs\profiles\rtt_2026_06__columns.csv
  outputs\profiles\rtt_2026_06__reporting_period.csv
  outputs\profiles\rtt_2026_06__provider.csv
  outputs\profiles\rtt_2026_06__provider_parent.csv
  outputs\profiles\rtt_2026_06__commissioner.csv
  outputs\profiles\rtt_2026_06__rtt_part.csv
  outputs\profiles\rtt_2026_06__treatment_function.csv
  outputs\profiles\rtt_2026_06__column_flags.csv
  outputs\profiles\rtt_2026_06__mapping_anomalies.csv
  outputs\profiles\rtt_2026_06__overview.md
  outputs\profiles\rtt_2026_06__MANIFEST.txt


## Structural safety checks

The profiler does not silently accept malformed structure. These lists are
empty for a well-formed file (they are, for June 2026).

In [3]:
print("structural_warnings          :", prof.structural_warnings or "none")
print("duplicate_headers            :", prof.duplicate_headers or "none")
print("unbound_roles                :", prof.unbound_roles or "none")
print("week_bucket_sequence_warnings:", prof.week_bucket_sequence_warnings or "none")
print("cells with embedded newline  :", prof.n_cells_with_embedded_newline)
print("parse truncated              :", prof.parse_truncated)

structural_warnings          : none
duplicate_headers            : none
unbound_roles                : none
week_bucket_sequence_warnings: none
cells with embedded newline  : 0
parse truncated              : False


## Rendered overview

The full Markdown overview saved to `outputs/profiles/`.

In [4]:
from IPython.display import Markdown, display

overview_md = PROFILE_DIR / f"{RAW_CSV.stem}__overview.md"
display(Markdown(overview_md.read_text(encoding="utf-8")))

# Data profile — `rtt_2026_06.csv`

- Generated (UTC): 2026-09-07T20:59:22+00:00
- Source path: `C:\Users\abdul\OneDrive\Desktop\P_Projects\NHS-RTT-Waiting-Times\data\raw\rtt_2026_06.csv`
- File size on disk: 78.3 MB (82,084,725 bytes)
- Encoding assumed: `utf-8-sig`  ·  delimiter: `,`
- Rows (excl. header): **182,411**
- Columns: **121**
- Duplicate rows (exact whole-row matches): **0**
- Ragged rows (field count != 121): **0**
- Estimated in-memory size as Python strings: ~797.8 MB (_estimate only — no pandas measurement_)
- Structural warnings: none

## Columns

`null_pct` is the percentage of the column that is empty **or whitespace-only**; literal `NULL`/`NA` text stays a string. `inferred_dtype` is a shape hint, not a schema. Blanks are **counted, not interpreted**.

| position | column | inferred_dtype | n_total | n_non_null | n_null | null_pct | n_unique | numeric_min | numeric_max | est_memory_bytes | type_mix |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 0 | Period | string | 182411 | 182411 | 0 | 0.0 | 1 | None | None | 11309482 | str=182411 |
| 1 | Provider Parent Org Code | string | 182411 | 182411 | 0 | 0.0 | 36 | None | None | 9592768 | str=182411 |
| 2 | Provider Parent Name | string | 182411 | 182411 | 0 | 0.0 | 36 | None | None | 17418821 | str=182411 |
| 3 | Provider Org Code | string | 182411 | 182411 | 0 | 0.0 | 537 | None | None | 9547226 | str=182411 |
| 4 | Provider Org Name | string | 182411 | 182411 | 0 | 0.0 | 536 | None | None | 16395083 | str=182411 |
| 5 | Commissioner Parent Org Code | string | 182411 | 172148 | 10263 | 5.6263 | 36 | None | None | 9047428 | str=172148;empty=10263 |
| 6 | Commissioner Parent Name | string | 182411 | 172148 | 10263 | 5.6263 | 36 | None | None | 16436662 | str=172148;empty=10263 |
| 7 | Commissioner Org Code | string | 182411 | 182411 | 0 | 0.0 | 129 | None | None | 9534739 | str=182411 |
| 8 | Commissioner Org Name | string | 182411 | 179063 | 3348 | 1.8354 | 121 | None | None | 14977268 | str=179063;empty=3348 |
| 9 | RTT Part Type | string | 182411 | 182411 | 0 | 0.0 | 5 | None | None | 10114307 | str=182411 |
| 10 | RTT Part Description | string | 182411 | 182411 | 0 | 0.0 | 5 | None | None | 14293412 | str=182411 |
| 11 | Treatment Function Code | string | 182411 | 182411 | 0 | 0.0 | 24 | None | None | 9790660 | str=182411 |
| 12 | Treatment Function Name | string | 182411 | 182411 | 0 | 0.0 | 24 | None | None | 12403280 | str=182411 |
| 13 | Gt 00 To 01 Weeks SUM 1 | integer | 182411 | 131561 | 50850 | 27.8766 | 769 | 0.0 | 6796.0 | 6592304 | int=131561;empty=50850 |
| 14 | Gt 01 To 02 Weeks SUM 1 | integer | 182411 | 130696 | 51715 | 28.3508 | 688 | 0.0 | 5364.0 | 6548001 | int=130696;empty=51715 |
| 15 | Gt 02 To 03 Weeks SUM 1 | integer | 182411 | 130598 | 51813 | 28.4045 | 649 | 0.0 | 5417.0 | 6542766 | int=130598;empty=51813 |
| 16 | Gt 03 To 04 Weeks SUM 1 | integer | 182411 | 130569 | 51842 | 28.4204 | 615 | 0.0 | 4939.0 | 6540891 | int=130569;empty=51842 |
| 17 | Gt 04 To 05 Weeks SUM 1 | integer | 182411 | 130350 | 52061 | 28.5405 | 570 | 0.0 | 4320.0 | 6529010 | int=130350;empty=52061 |
| 18 | Gt 05 To 06 Weeks SUM 1 | integer | 182411 | 130160 | 52251 | 28.6447 | 521 | 0.0 | 3675.0 | 6518460 | int=130160;empty=52251 |
| 19 | Gt 06 To 07 Weeks SUM 1 | integer | 182411 | 130316 | 52095 | 28.5591 | 553 | 0.0 | 4515.0 | 6527052 | int=130316;empty=52095 |
| 20 | Gt 07 To 08 Weeks SUM 1 | integer | 182411 | 130153 | 52258 | 28.6485 | 547 | 0.0 | 4159.0 | 6518303 | int=130153;empty=52258 |
| 21 | Gt 08 To 09 Weeks SUM 1 | integer | 182411 | 129755 | 52656 | 28.8667 | 475 | 0.0 | 3225.0 | 6496887 | int=129755;empty=52656 |
| 22 | Gt 09 To 10 Weeks SUM 1 | integer | 182411 | 129800 | 52611 | 28.842 | 501 | 0.0 | 3851.0 | 6499492 | int=129800;empty=52611 |
| 23 | Gt 10 To 11 Weeks SUM 1 | integer | 182411 | 129631 | 52780 | 28.9347 | 492 | 0.0 | 3569.0 | 6490626 | int=129631;empty=52780 |
| 24 | Gt 11 To 12 Weeks SUM 1 | integer | 182411 | 129502 | 52909 | 29.0054 | 463 | 0.0 | 3177.0 | 6483510 | int=129502;empty=52909 |
| 25 | Gt 12 To 13 Weeks SUM 1 | integer | 182411 | 129053 | 53358 | 29.2515 | 389 | 0.0 | 2233.0 | 6459332 | int=129053;empty=53358 |
| 26 | Gt 13 To 14 Weeks SUM 1 | integer | 182411 | 129424 | 52987 | 29.0481 | 463 | 0.0 | 3373.0 | 6479365 | int=129424;empty=52987 |
| 27 | Gt 14 To 15 Weeks SUM 1 | integer | 182411 | 129304 | 53107 | 29.1139 | 441 | 0.0 | 3109.0 | 6472987 | int=129304;empty=53107 |
| 28 | Gt 15 To 16 Weeks SUM 1 | integer | 182411 | 129250 | 53161 | 29.1435 | 440 | 0.0 | 2998.0 | 6470226 | int=129250;empty=53161 |
| 29 | Gt 16 To 17 Weeks SUM 1 | integer | 182411 | 129127 | 53284 | 29.211 | 440 | 0.0 | 3011.0 | 6463787 | int=129127;empty=53284 |
| 30 | Gt 17 To 18 Weeks SUM 1 | integer | 182411 | 129105 | 53306 | 29.223 | 424 | 0.0 | 3101.0 | 6462490 | int=129105;empty=53306 |
| 31 | Gt 18 To 19 Weeks SUM 1 | integer | 182411 | 128710 | 53701 | 29.4396 | 385 | 0.0 | 2321.0 | 6441818 | int=128710;empty=53701 |
| 32 | Gt 19 To 20 Weeks SUM 1 | integer | 182411 | 128734 | 53677 | 29.4264 | 381 | 0.0 | 2291.0 | 6443059 | int=128734;empty=53677 |
| 33 | Gt 20 To 21 Weeks SUM 1 | integer | 182411 | 128640 | 53771 | 29.4779 | 383 | 0.0 | 2332.0 | 6438230 | int=128640;empty=53771 |
| 34 | Gt 21 To 22 Weeks SUM 1 | integer | 182411 | 128556 | 53855 | 29.524 | 369 | 0.0 | 2316.0 | 6433694 | int=128556;empty=53855 |
| 35 | Gt 22 To 23 Weeks SUM 1 | integer | 182411 | 128319 | 54092 | 29.6539 | 363 | 0.0 | 2155.0 | 6421430 | int=128319;empty=54092 |
| 36 | Gt 23 To 24 Weeks SUM 1 | integer | 182411 | 128219 | 54192 | 29.7087 | 348 | 0.0 | 2023.0 | 6416129 | int=128219;empty=54192 |
| 37 | Gt 24 To 25 Weeks SUM 1 | integer | 182411 | 128167 | 54244 | 29.7372 | 335 | 0.0 | 1920.0 | 6413284 | int=128167;empty=54244 |
| 38 | Gt 25 To 26 Weeks SUM 1 | integer | 182411 | 127709 | 54702 | 29.9883 | 262 | 0.0 | 1095.0 | 6389096 | int=127709;empty=54702 |
| 39 | Gt 26 To 27 Weeks SUM 1 | integer | 182411 | 127423 | 54988 | 30.1451 | 229 | 0.0 | 864.0 | 6374142 | int=127423;empty=54988 |
| 40 | Gt 27 To 28 Weeks SUM 1 | integer | 182411 | 127977 | 54434 | 29.8414 | 300 | 0.0 | 1579.0 | 6403413 | int=127977;empty=54434 |
| 41 | Gt 28 To 29 Weeks SUM 1 | integer | 182411 | 127994 | 54417 | 29.8321 | 315 | 0.0 | 1569.0 | 6404283 | int=127994;empty=54417 |
| 42 | Gt 29 To 30 Weeks SUM 1 | integer | 182411 | 127888 | 54523 | 29.8902 | 302 | 0.0 | 1452.0 | 6398791 | int=127888;empty=54523 |
| 43 | Gt 30 To 31 Weeks SUM 1 | integer | 182411 | 127818 | 54593 | 29.9286 | 303 | 0.0 | 1365.0 | 6395172 | int=127818;empty=54593 |
| 44 | Gt 31 To 32 Weeks SUM 1 | integer | 182411 | 127767 | 54644 | 29.9565 | 292 | 0.0 | 1404.0 | 6392527 | int=127767;empty=54644 |
| 45 | Gt 32 To 33 Weeks SUM 1 | integer | 182411 | 127775 | 54636 | 29.9521 | 286 | 0.0 | 1251.0 | 6392844 | int=127775;empty=54636 |
| 46 | Gt 33 To 34 Weeks SUM 1 | integer | 182411 | 127597 | 54814 | 30.0497 | 277 | 0.0 | 1184.0 | 6383756 | int=127597;empty=54814 |
| 47 | Gt 34 To 35 Weeks SUM 1 | integer | 182411 | 127553 | 54858 | 30.0738 | 267 | 0.0 | 1093.0 | 6381275 | int=127553;empty=54858 |
| 48 | Gt 35 To 36 Weeks SUM 1 | integer | 182411 | 127451 | 54960 | 30.1298 | 262 | 0.0 | 1060.0 | 6376135 | int=127451;empty=54960 |
| 49 | Gt 36 To 37 Weeks SUM 1 | integer | 182411 | 127459 | 54952 | 30.1254 | 266 | 0.0 | 1095.0 | 6376486 | int=127459;empty=54952 |
| 50 | Gt 37 To 38 Weeks SUM 1 | integer | 182411 | 127418 | 54993 | 30.1479 | 248 | 0.0 | 1028.0 | 6374392 | int=127418;empty=54993 |
| 51 | Gt 38 To 39 Weeks SUM 1 | integer | 182411 | 127348 | 55063 | 30.1862 | 256 | 0.0 | 861.0 | 6370778 | int=127348;empty=55063 |
| 52 | Gt 39 To 40 Weeks SUM 1 | integer | 182411 | 127210 | 55201 | 30.2619 | 235 | 0.0 | 851.0 | 6363607 | int=127210;empty=55201 |
| 53 | Gt 40 To 41 Weeks SUM 1 | integer | 182411 | 127138 | 55273 | 30.3014 | 233 | 0.0 | 797.0 | 6359889 | int=127138;empty=55273 |
| 54 | Gt 41 To 42 Weeks SUM 1 | integer | 182411 | 127136 | 55275 | 30.3024 | 234 | 0.0 | 752.0 | 6359718 | int=127136;empty=55275 |
| 55 | Gt 42 To 43 Weeks SUM 1 | integer | 182411 | 127027 | 55384 | 30.3622 | 235 | 0.0 | 696.0 | 6354119 | int=127027;empty=55384 |
| 56 | Gt 43 To 44 Weeks SUM 1 | integer | 182411 | 126946 | 55465 | 30.4066 | 204 | 0.0 | 625.0 | 6349878 | int=126946;empty=55465 |
| 57 | Gt 44 To 45 Weeks SUM 1 | integer | 182411 | 126763 | 55648 | 30.5069 | 181 | 0.0 | 519.0 | 6340197 | int=126763;empty=55648 |
| 58 | Gt 45 To 46 Weeks SUM 1 | integer | 182411 | 126779 | 55632 | 30.4982 | 202 | 0.0 | 589.0 | 6341331 | int=126779;empty=55632 |
| 59 | Gt 46 To 47 Weeks SUM 1 | integer | 182411 | 126797 | 55614 | 30.4883 | 191 | 0.0 | 566.0 | 6342143 | int=126797;empty=55614 |
| 60 | Gt 47 To 48 Weeks SUM 1 | integer | 182411 | 126723 | 55688 | 30.5289 | 186 | 0.0 | 540.0 | 6338280 | int=126723;empty=55688 |
| 61 | Gt 48 To 49 Weeks SUM 1 | integer | 182411 | 126627 | 55784 | 30.5815 | 180 | 0.0 | 590.0 | 6333332 | int=126627;empty=55784 |
| 62 | Gt 49 To 50 Weeks SUM 1 | integer | 182411 | 126551 | 55860 | 30.6232 | 171 | 0.0 | 463.0 | 6329450 | int=126551;empty=55860 |
| 63 | Gt 50 To 51 Weeks SUM 1 | integer | 182411 | 126527 | 55884 | 30.6363 | 164 | 0.0 | 482.0 | 6328150 | int=126527;empty=55884 |
| 64 | Gt 51 To 52 Weeks SUM 1 | integer | 182411 | 126450 | 55961 | 30.6785 | 153 | 0.0 | 461.0 | 6324117 | int=126450;empty=55961 |
| 65 | Gt 52 To 53 Weeks SUM 1 | integer | 182411 | 126199 | 56212 | 30.8161 | 134 | 0.0 | 395.0 | 6311290 | int=126199;empty=56212 |
| 66 | Gt 53 To 54 Weeks SUM 1 | integer | 182411 | 126097 | 56314 | 30.872 | 123 | 0.0 | 336.0 | 6306012 | int=126097;empty=56314 |
| 67 | Gt 54 To 55 Weeks SUM 1 | integer | 182411 | 126019 | 56392 | 30.9148 | 120 | 0.0 | 307.0 | 6301996 | int=126019;empty=56392 |
| 68 | Gt 55 To 56 Weeks SUM 1 | integer | 182411 | 125897 | 56514 | 30.9817 | 114 | 0.0 | 267.0 | 6295813 | int=125897;empty=56514 |
| 69 | Gt 56 To 57 Weeks SUM 1 | integer | 182411 | 125493 | 56918 | 31.2032 | 101 | 0.0 | 237.0 | 6275437 | int=125493;empty=56918 |
| 70 | Gt 57 To 58 Weeks SUM 1 | integer | 182411 | 125731 | 56680 | 31.0727 | 89 | 0.0 | 206.0 | 6287154 | int=125731;empty=56680 |
| 71 | Gt 58 To 59 Weeks SUM 1 | integer | 182411 | 125444 | 56967 | 31.23 | 91 | 0.0 | 195.0 | 6272821 | int=125444;empty=56967 |
| 72 | Gt 59 To 60 Weeks SUM 1 | integer | 182411 | 125665 | 56746 | 31.1089 | 89 | 0.0 | 213.0 | 6283824 | int=125665;empty=56746 |
| 73 | Gt 60 To 61 Weeks SUM 1 | integer | 182411 | 124980 | 57431 | 31.4844 | 65 | 0.0 | 110.0 | 6249371 | int=124980;empty=57431 |
| 74 | Gt 61 To 62 Weeks SUM 1 | integer | 182411 | 125517 | 56894 | 31.19 | 64 | 0.0 | 167.0 | 6276236 | int=125517;empty=56894 |
| 75 | Gt 62 To 63 Weeks SUM 1 | integer | 182411 | 124847 | 57564 | 31.5573 | 53 | 0.0 | 75.0 | 6242602 | int=124847;empty=57564 |
| 76 | Gt 63 To 64 Weeks SUM 1 | integer | 182411 | 124870 | 57541 | 31.5447 | 59 | 0.0 | 123.0 | 6243807 | int=124870;empty=57541 |
| 77 | Gt 64 To 65 Weeks SUM 1 | integer | 182411 | 125390 | 57021 | 31.2596 | 52 | 0.0 | 102.0 | 6269763 | int=125390;empty=57021 |
| 78 | Gt 65 To 66 Weeks SUM 1 | integer | 182411 | 125232 | 57179 | 31.3462 | 42 | 0.0 | 86.0 | 6261748 | int=125232;empty=57179 |
| 79 | Gt 66 To 67 Weeks SUM 1 | integer | 182411 | 124315 | 58096 | 31.849 | 44 | 0.0 | 77.0 | 6215859 | int=124315;empty=58096 |
| 80 | Gt 67 To 68 Weeks SUM 1 | integer | 182411 | 124483 | 57928 | 31.7569 | 35 | 0.0 | 79.0 | 6224232 | int=124483;empty=57928 |
| 81 | Gt 68 To 69 Weeks SUM 1 | integer | 182411 | 124258 | 58153 | 31.8802 | 32 | 0.0 | 85.0 | 6212973 | int=124258;empty=58153 |
| 82 | Gt 69 To 70 Weeks SUM 1 | integer | 182411 | 123947 | 58464 | 32.0507 | 31 | 0.0 | 59.0 | 6197401 | int=123947;empty=58464 |
| 83 | Gt 70 To 71 Weeks SUM 1 | integer | 182411 | 124242 | 58169 | 31.889 | 25 | 0.0 | 43.0 | 6212136 | int=124242;empty=58169 |
| 84 | Gt 71 To 72 Weeks SUM 1 | integer | 182411 | 124020 | 58391 | 32.0107 | 28 | 0.0 | 56.0 | 6201032 | int=124020;empty=58391 |
| 85 | Gt 72 To 73 Weeks SUM 1 | integer | 182411 | 124205 | 58206 | 31.9093 | 24 | 0.0 | 49.0 | 6210275 | int=124205;empty=58206 |
| 86 | Gt 73 To 74 Weeks SUM 1 | integer | 182411 | 123993 | 58418 | 32.0255 | 21 | 0.0 | 39.0 | 6199671 | int=123993;empty=58418 |
| 87 | Gt 74 To 75 Weeks SUM 1 | integer | 182411 | 123502 | 58909 | 32.2947 | 20 | 0.0 | 45.0 | 6175113 | int=123502;empty=58909 |
| 88 | Gt 75 To 76 Weeks SUM 1 | integer | 182411 | 123876 | 58535 | 32.0896 | 23 | 0.0 | 56.0 | 6193819 | int=123876;empty=58535 |
| 89 | Gt 76 To 77 Weeks SUM 1 | integer | 182411 | 123800 | 58611 | 32.1313 | 20 | 0.0 | 50.0 | 6190014 | int=123800;empty=58611 |
| 90 | Gt 77 To 78 Weeks SUM 1 | integer | 182411 | 123865 | 58546 | 32.0957 | 16 | 0.0 | 37.0 | 6193258 | int=123865;empty=58546 |
| 91 | Gt 78 To 79 Weeks SUM 1 | integer | 182411 | 123481 | 58930 | 32.3062 | 11 | 0.0 | 17.0 | 6174052 | int=123481;empty=58930 |
| 92 | Gt 79 To 80 Weeks SUM 1 | integer | 182411 | 123797 | 58614 | 32.1329 | 12 | 0.0 | 31.0 | 6189855 | int=123797;empty=58614 |
| 93 | Gt 80 To 81 Weeks SUM 1 | integer | 182411 | 122187 | 60224 | 33.0156 | 13 | 0.0 | 41.0 | 6109358 | int=122187;empty=60224 |
| 94 | Gt 81 To 82 Weeks SUM 1 | integer | 182411 | 123483 | 58928 | 32.3051 | 13 | 0.0 | 31.0 | 6174156 | int=123483;empty=58928 |
| 95 | Gt 82 To 83 Weeks SUM 1 | integer | 182411 | 123493 | 58918 | 32.2996 | 13 | 0.0 | 40.0 | 6174654 | int=123493;empty=58918 |
| 96 | Gt 83 To 84 Weeks SUM 1 | integer | 182411 | 122167 | 60244 | 33.0265 | 15 | 0.0 | 41.0 | 6108356 | int=122167;empty=60244 |
| 97 | Gt 84 To 85 Weeks SUM 1 | integer | 182411 | 122172 | 60239 | 33.0238 | 13 | 0.0 | 32.0 | 6108604 | int=122172;empty=60239 |
| 98 | Gt 85 To 86 Weeks SUM 1 | integer | 182411 | 122170 | 60241 | 33.0249 | 11 | 0.0 | 34.0 | 6108504 | int=122170;empty=60241 |
| 99 | Gt 86 To 87 Weeks SUM 1 | integer | 182411 | 123477 | 58934 | 32.3084 | 12 | 0.0 | 21.0 | 6173855 | int=123477;empty=58934 |
| 100 | Gt 87 To 88 Weeks SUM 1 | integer | 182411 | 123486 | 58925 | 32.3034 | 10 | 0.0 | 33.0 | 6174304 | int=123486;empty=58925 |
| 101 | Gt 88 To 89 Weeks SUM 1 | integer | 182411 | 122163 | 60248 | 33.0287 | 11 | 0.0 | 44.0 | 6108154 | int=122163;empty=60248 |
| 102 | Gt 89 To 90 Weeks SUM 1 | integer | 182411 | 122164 | 60247 | 33.0282 | 12 | 0.0 | 15.0 | 6108204 | int=122164;empty=60247 |
| 103 | Gt 90 To 91 Weeks SUM 1 | integer | 182411 | 122163 | 60248 | 33.0287 | 9 | 0.0 | 39.0 | 6108154 | int=122163;empty=60248 |
| 104 | Gt 91 To 92 Weeks SUM 1 | integer | 182411 | 123487 | 58924 | 32.3029 | 10 | 0.0 | 26.0 | 6174358 | int=123487;empty=58924 |
| 105 | Gt 92 To 93 Weeks SUM 1 | integer | 182411 | 122165 | 60246 | 33.0276 | 8 | 0.0 | 31.0 | 6108254 | int=122165;empty=60246 |
| 106 | Gt 93 To 94 Weeks SUM 1 | integer | 182411 | 122168 | 60243 | 33.026 | 9 | 0.0 | 19.0 | 6108404 | int=122168;empty=60243 |
| 107 | Gt 94 To 95 Weeks SUM 1 | integer | 182411 | 123478 | 58933 | 32.3078 | 10 | 0.0 | 35.0 | 6173906 | int=123478;empty=58933 |
| 108 | Gt 95 To 96 Weeks SUM 1 | integer | 182411 | 122160 | 60251 | 33.0304 | 10 | 0.0 | 26.0 | 6108004 | int=122160;empty=60251 |
| 109 | Gt 96 To 97 Weeks SUM 1 | integer | 182411 | 122163 | 60248 | 33.0287 | 6 | 0.0 | 10.0 | 6108152 | int=122163;empty=60248 |
| 110 | Gt 97 To 98 Weeks SUM 1 | integer | 182411 | 123476 | 58935 | 32.3089 | 8 | 0.0 | 16.0 | 6173804 | int=123476;empty=58935 |
| 111 | Gt 98 To 99 Weeks SUM 1 | integer | 182411 | 123480 | 58931 | 32.3067 | 8 | 0.0 | 8.0 | 6174000 | int=123480;empty=58931 |
| 112 | Gt 99 To 100 Weeks SUM 1 | integer | 182411 | 122167 | 60244 | 33.0265 | 6 | 0.0 | 13.0 | 6108352 | int=122167;empty=60244 |
| 113 | Gt 100 To 101 Weeks SUM 1 | integer | 182411 | 122162 | 60249 | 33.0293 | 7 | 0.0 | 20.0 | 6108104 | int=122162;empty=60249 |
| 114 | Gt 101 To 102 Weeks SUM 1 | integer | 182411 | 123785 | 58626 | 32.1395 | 7 | 0.0 | 13.0 | 6189252 | int=123785;empty=58626 |
| 115 | Gt 102 To 103 Weeks SUM 1 | integer | 182411 | 123478 | 58933 | 32.3078 | 6 | 0.0 | 14.0 | 6173902 | int=123478;empty=58933 |
| 116 | Gt 103 To 104 Weeks SUM 1 | integer | 182411 | 123474 | 58937 | 32.31 | 7 | 0.0 | 11.0 | 6173702 | int=123474;empty=58937 |
| 117 | Gt 104 Weeks SUM 1 | integer | 182411 | 123744 | 58667 | 32.162 | 43 | 0.0 | 367.0 | 6187266 | int=123744;empty=58667 |
| 118 | Total | integer | 182411 | 51154 | 131257 | 71.9567 | 1283 | 0.0 | 19389.0 | 2577983 | empty=131257;int=51154 |
| 119 | Patients with unknown clock start date | integer | 182411 | 34117 | 148294 | 81.2966 | 30 | 0.0 | 249.0 | 1705888 | empty=148294;int=34117 |
| 120 | Total All | integer | 182411 | 182411 | 0 | 0.0 | 3576 | 1.0 | 108631.0 | 9198651 | int=182411 |


## Column flags (header text only)

Candidate groupings surfaced for review. Membership is decided purely from the header string; **no semantic meaning is assigned** here. A column may appear under more than one flag. Sequence integrity of the week buckets is checked separately (see structural warnings, if any).

- **total_columns** (2): `Total`, `Total All`
- **unknown_columns** (1): `Patients with unknown clock start date`
- **week_bucket_columns** (105): `Gt 00 To 01 Weeks SUM 1` … `Gt 104 Weeks SUM 1` (full list in `rtt_2026_06__column_flags.csv`)


## reporting_period — 1 distinct code(s)

Source columns: `Period`

| code | row_count |
| --- | --- |
| RTT-June-2026 | 182411 |


## provider — 537 distinct code(s), 536 distinct name(s)

Source columns: `Provider Org Code` / `Provider Org Name`

**Mapping is not 1:1** — 1 anomaly row(s). A blank-vs-blank pair is reported here too; this does not pick a correct value:

| kind | key | n_related | related_values |
| --- | --- | --- | --- |
| name_has_multiple_codes | DUCHY HOSPITAL | 2 | NT447 \| NVC04 |


| code | row_count | name |
| --- | --- | --- |
| A0C5S | 22 | SPAMEDICA GLOUCESTER |
| A1D1B | 54 | SPAMEDICA WEMBLEY |
| A1S9E | 38 | OPTEGRA EYE HOSPITAL YORK |
| A1U4J | 36 | SPAMEDICA PORTSMOUTH |
| A4M8P | 287 | BUCKSHAW HOSPITAL |
| A4Q9X | 60 | NEWMEDICA LEICESTER |
| A4Y8X | 28 | ACES NOTTINGHAM |
| A5E1F | 64 | OPTEGRA EYE HOSPITAL PRESTON |
| A6U7U | 36 | OPTEGRA EYE HOSPITAL NOTTINGHAM |
| A8L1O | 34 | OPTEGRA EYE HOSPITAL LEICESTER |
| A9P5F | 62 | ACES WHITE CITY |
| A9T5Y | 14 | RSA (ATHENA SURGICAL CENTRE) |
| AAH | 121 | TETBURY HOSPITAL TRUST LTD |
| AAV | 770 | COMMUNITY HEALTH AND EYECARE LIMITED |
| ACG08 | 8 | NEWMEDICA OPHTHALMOLOGY - NUNEATON - CAMP HILL - GP LED HEALTH CENTRE |
| ACG12 | 52 | NEWMEDICA COMMUNITY OPHTHALMOLOGY - NORTH EAST LINCOLNSHIRE - GRIMSBY |
| ACG13 | 36 | NEWMEDICA - GLOUCESTER (ASPEN MEDICAL CENTRE) |
| ACG19 | 68 | NEWMEDICA COMMUNITY OPHTHALMOLOGY - LEEDS |
| ACG20 | 38 | NEWMEDICA - BRISTOL - LITFIELD HOUSE |
| ACG22 | 78 | NEWMEDICA - BRIGG - RIVERSIDE SURGERY |
| ACG24 | 62 | NEWMEDICA - TEESSIDE - NORTH ORMESBY |
| ACG26 | 94 | NEWMEDICA - NORTH DERBYSHIRE - MIDLAND COURT |
| ACG30 | 60 | NEWMEDICA - WORKSOP - DUKERIES |
| ACG31 | 84 | NEWMEDICA - WORCESTER |
| ACG36 | 34 | NEWMEDICA - EXETER - GLEN HOUSE |
| ACG37 | 68 | NEWMEDICA - IPSWICH |
| ACG38 | 32 | NEWMEDICA - LANGFORD |
| ACG40 | 66 | NEWMEDICA - WAKEFIELD |
| ACM | 12 | EYE CARE MEDICAL LTD |
| ADP02 | 42 | KIMS HOSPITAL (NEWNHAM COURT) |
| AHP | 6 | ST MICHAEL'S CLINIC LIMITED |
| AJ8 | 10 | THE GRANGE MEDICAL CENTRE HQ |
| AQK | 9 | VERNOVA HEALTHCARE COMMUNITY INTEREST COMPANY |
| AVQ01 | 22 | ONE ASHFORD HOSPITAL |
| AVQ03 | 38 | ONE HATFIELD HOSPITAL |
| AW7 | 18 | REGENCY EYE HOSPITAL |
| AWR | 89 | ONE STOP DOCTORS LIMITED |
| AY1 | 67 | MEDEFER |
| B1A5K | 72 | NEWMEDICA - NOTTINGHAM |
| B3M1X | 322 | CLAREMONT PRIVATE HOSPITAL |
| B4N1U | 74 | SPAMEDICA ROMFORD |
| B5M7B | 80 | NEWMEDICA - NORWICH |
| B6Z0N | 86 | OPTEGRA EYE HOSPITAL UTTOXETER |
| B9J4U | 54 | SPAMEDICA PETERBOROUGH |
| B9M3W | 44 | SPAMEDICA SOUTHAMPTON |
| C3Y0A | 30 | SPAMEDICA BLACKPOOL |
| C5Y9L | 54 | NEWMEDICA - BERKSHIRE - BRACKNELL |
| C9A7J | 64 | NEWMEDICA - BRADFORD & HUDDERSFIELD |
| D0V5X | 26 | NEWMEDICA - SWINDON MURDOCK ROAD |
| D0Y2J | 14 | OPTEGRA AT HAVANT HEALTH CENTRE (HAVANT) |
| D1V5N | 2 | ACES BOLTON |
| D2L5T | 40 | ACES LIVERPOOL |
| D3O0Z | 22 | PLYMOUTH NEWMEDICA LTD |
| D3R0C | 2 | ACES CARLISLE |
| D5P5H | 26 | SPAMEDICA BEXHILL |
| D9E5L | 134 | OPTEGRA EYE HOSPITAL BRIGHTON |
| DFQ | 23 | NORWICH & NORFOLK SURGICAL LTD (N2S) |
| DFY01 | 35 | COTSWOLDS SURGICAL PARTNERS |
| DJH01 | 12 | THE ROYAL BUCKINGHAMSHIRE HOSPITAL LTD |
| DPK | 34 | THE STONEYGATE EYE HOSPITAL |


_… 477 more not shown; see the CSV._


## provider_parent — 36 distinct code(s), 36 distinct name(s)

Source columns: `Provider Parent Org Code` / `Provider Parent Name`

Code/name mapping is 1:1 across the raw values in this file.

| code | row_count | name |
| --- | --- | --- |
| D7T5G | 5371 | NHS ESSEX INTEGRATED CARE BOARD |
| QE1 | 7167 | NHS LANCASHIRE AND SOUTH CUMBRIA INTEGRATED CARE BOARD |
| QF7 | 4626 | NHS SOUTH YORKSHIRE INTEGRATED CARE BOARD |
| QGH | 2108 | NHS HEREFORDSHIRE AND WORCESTERSHIRE INTEGRATED CARE BOARD |
| QHL | 5058 | NHS BIRMINGHAM AND SOLIHULL INTEGRATED CARE BOARD |
| QHM | 8348 | NHS NORTH EAST AND NORTH CUMBRIA INTEGRATED CARE BOARD |
| QJ2 | 2673 | NHS DERBY AND DERBYSHIRE INTEGRATED CARE BOARD |
| QJK | 2153 | NHS DEVON INTEGRATED CARE BOARD |
| QJM | 1158 | NHS LINCOLNSHIRE INTEGRATED CARE BOARD |
| QK1 | 2220 | NHS LEICESTER, LEICESTERSHIRE AND RUTLAND INTEGRATED CARE BOARD |
| QKK | 8525 | NHS SOUTH EAST LONDON INTEGRATED CARE BOARD |
| QKS | 3402 | NHS KENT AND MEDWAY INTEGRATED CARE BOARD |
| QMF | 5922 | NHS NORTH EAST LONDON INTEGRATED CARE BOARD |
| QNC | 2812 | NHS STAFFORDSHIRE AND STOKE-ON-TRENT INTEGRATED CARE BOARD |
| QOC | 1759 | NHS SHROPSHIRE, TELFORD AND WREKIN INTEGRATED CARE BOARD |
| QOP | 13761 | NHS GREATER MANCHESTER INTEGRATED CARE BOARD |
| QOQ | 5530 | NHS HUMBER AND NORTH YORKSHIRE INTEGRATED CARE BOARD |
| QOX | 3668 | NHS BATH AND NORTH EAST SOMERSET, SWINDON AND WILTSHIRE INTEGRATED CARE BOARD |
| QPM | 1901 | NHS NORTHAMPTONSHIRE INTEGRATED CARE BOARD |
| QR1 | 1302 | NHS GLOUCESTERSHIRE INTEGRATED CARE BOARD |
| QRL | 4794 | NHS HAMPSHIRE AND ISLE OF WIGHT INTEGRATED CARE BOARD |
| QSL | 1119 | NHS SOMERSET INTEGRATED CARE BOARD |
| QT1 | 2837 | NHS NOTTINGHAM AND NOTTINGHAMSHIRE INTEGRATED CARE BOARD |
| QT6 | 640 | NHS CORNWALL AND THE ISLES OF SCILLY INTEGRATED CARE BOARD |
| QUA | 4777 | NHS BLACK COUNTRY INTEGRATED CARE BOARD |
| QUY | 2812 | NHS BRISTOL, NORTH SOMERSET AND SOUTH GLOUCESTERSHIRE INTEGRATED CARE BOARD |
| QVV | 1622 | NHS DORSET INTEGRATED CARE BOARD |
| QWE | 3962 | NHS SOUTH WEST LONDON INTEGRATED CARE BOARD |
| QWO | 7697 | NHS WEST YORKSHIRE INTEGRATED CARE BOARD |
| QWU | 3257 | NHS COVENTRY AND WARWICKSHIRE INTEGRATED CARE BOARD |
| QYG | 11103 | NHS CHESHIRE AND MERSEYSIDE INTEGRATED CARE BOARD |
| S0E4D | 6012 | NHS THAMES VALLEY INTEGRATED CARE BOARD |
| S1Y5D | 10719 | NHS CENTRAL EAST INTEGRATED CARE BOARD |
| S9B9J | 8491 | NHS SURREY AND SUSSEX INTEGRATED CARE BOARD |
| T6Y0W | 3219 | NHS NORFOLK AND SUFFOLK INTEGRATED CARE BOARD |
| Z9B2Z | 19886 | NHS WEST AND NORTH LONDON INTEGRATED CARE BOARD |


## commissioner — 129 distinct code(s), 121 distinct name(s) (+1 blank/whitespace)

Source columns: `Commissioner Org Code` / `Commissioner Org Name`

**Mapping is not 1:1** — 1 anomaly row(s). A blank-vs-blank pair is reported here too; this does not pick a correct value:

| kind | key | n_related | related_values |
| --- | --- | --- | --- |
| name_has_multiple_codes |  | 8 | NONC \| Y56 \| Y58 \| Y59 \| Y60 \| Y61 \| Y62 \| Y63 |


| code | row_count | name |
| --- | --- | --- |
| 00L | 826 | NHS NORTH EAST AND NORTH CUMBRIA ICB - 00L |
| 00N | 634 | NHS NORTH EAST AND NORTH CUMBRIA ICB - 00N |
| 00P | 805 | NHS NORTH EAST AND NORTH CUMBRIA ICB - 00P |
| 00Q | 896 | NHS LANCASHIRE AND SOUTH CUMBRIA ICB - 00Q |
| 00R | 1012 | NHS LANCASHIRE AND SOUTH CUMBRIA ICB - 00R |
| 00T | 1152 | NHS GREATER MANCHESTER ICB - 00T |
| 00V | 948 | NHS GREATER MANCHESTER ICB - 00V |
| 00X | 1207 | NHS LANCASHIRE AND SOUTH CUMBRIA ICB - 00X |
| 00Y | 965 | NHS GREATER MANCHESTER ICB - 00Y |
| 01A | 1548 | NHS LANCASHIRE AND SOUTH CUMBRIA ICB - 01A |
| 01D | 928 | NHS GREATER MANCHESTER ICB - 01D |
| 01E | 1188 | NHS LANCASHIRE AND SOUTH CUMBRIA ICB - 01E |
| 01F | 824 | NHS CHESHIRE AND MERSEYSIDE ICB - 01F |
| 01G | 1280 | NHS GREATER MANCHESTER ICB - 01G |
| 01H | 1269 | NHS NORTH EAST AND NORTH CUMBRIA ICB - 01H |
| 01J | 766 | NHS CHESHIRE AND MERSEYSIDE ICB - 01J |
| 01K | 1375 | NHS LANCASHIRE AND SOUTH CUMBRIA ICB - 01K |
| 01T | 744 | NHS CHESHIRE AND MERSEYSIDE ICB - 01T |
| 01V | 747 | NHS CHESHIRE AND MERSEYSIDE ICB - 01V |
| 01W | 1238 | NHS GREATER MANCHESTER ICB - 01W |
| 01X | 986 | NHS CHESHIRE AND MERSEYSIDE ICB - 01X |
| 01Y | 1006 | NHS GREATER MANCHESTER ICB - 01Y |
| 02A | 1093 | NHS GREATER MANCHESTER ICB - 02A |
| 02E | 1241 | NHS CHESHIRE AND MERSEYSIDE ICB - 02E |
| 02G | 983 | NHS LANCASHIRE AND SOUTH CUMBRIA ICB - 02G |
| 02H | 1321 | NHS GREATER MANCHESTER ICB - 02H |
| 02M | 1158 | NHS LANCASHIRE AND SOUTH CUMBRIA ICB - 02M |
| 02P | 1059 | NHS SOUTH YORKSHIRE ICB - 02P |
| 02Q | 898 | NHS NOTTINGHAM AND NOTTINGHAMSHIRE ICB - 02Q |
| 02T | 1071 | NHS WEST YORKSHIRE ICB - 02T |
| 02X | 1187 | NHS SOUTH YORKSHIRE ICB - 02X |
| 02Y | 1259 | NHS HUMBER AND NORTH YORKSHIRE ICB - 02Y |
| 03F | 858 | NHS HUMBER AND NORTH YORKSHIRE ICB - 03F |
| 03H | 863 | NHS HUMBER AND NORTH YORKSHIRE ICB - 03H |
| 03K | 1017 | NHS HUMBER AND NORTH YORKSHIRE ICB - 03K |
| 03L | 962 | NHS SOUTH YORKSHIRE ICB - 03L |
| 03N | 1487 | NHS SOUTH YORKSHIRE ICB - 03N |
| 03Q | 1363 | NHS HUMBER AND NORTH YORKSHIRE ICB - 03Q |
| 03R | 1382 | NHS WEST YORKSHIRE ICB - 03R |
| 03W | 1733 | NHS LEICESTER, LEICESTERSHIRE AND RUTLAND ICB - 03W |
| 04C | 1171 | NHS LEICESTER, LEICESTERSHIRE AND RUTLAND ICB - 04C |
| 04V | 1773 | NHS LEICESTER, LEICESTERSHIRE AND RUTLAND ICB - 04V |
| 04Y | 958 | NHS STAFFORDSHIRE AND STOKE-ON-TRENT ICB - 04Y |
| 05D | 1048 | NHS STAFFORDSHIRE AND STOKE-ON-TRENT ICB - 05D |
| 05G | 1134 | NHS STAFFORDSHIRE AND STOKE-ON-TRENT ICB - 05G |
| 05Q | 1522 | NHS STAFFORDSHIRE AND STOKE-ON-TRENT ICB - 05Q |
| 05V | 1008 | NHS STAFFORDSHIRE AND STOKE-ON-TRENT ICB - 05V |
| 05W | 1014 | NHS STAFFORDSHIRE AND STOKE-ON-TRENT ICB - 05W |
| 06H | 2558 | NHS CENTRAL EAST ICB - 06H |
| 06K | 2228 | NHS CENTRAL EAST ICB - 06K |
| 06L | 1410 | NHS NORFOLK AND SUFFOLK ICB - 06L |
| 06N | 2307 | NHS CENTRAL EAST ICB - 06N |
| 06Q | 1698 | NHS ESSEX ICB - 06Q |
| 06T | 1343 | NHS ESSEX ICB - 06T |
| 07G | 1273 | NHS ESSEX ICB - 07G |
| 07H | 1893 | NHS ESSEX ICB - 07H |
| 07K | 1198 | NHS NORFOLK AND SUFFOLK ICB - 07K |
| 09D | 1447 | NHS SURREY AND SUSSEX ICB - 09D |
| 10Q | 2120 | NHS THAMES VALLEY ICB - 10Q |
| 10R | 884 | NHS HAMPSHIRE AND ISLE OF WIGHT ICB - 10R |


_… 69 more not shown; see the CSV._


## rtt_part — 5 distinct code(s), 5 distinct name(s)

Source columns: `RTT Part Type` / `RTT Part Description`

Code/name mapping is 1:1 across the raw values in this file.

| code | row_count | name |
| --- | --- | --- |
| Part_1A | 18803 | Completed Pathways For Admitted Patients |
| Part_1B | 32351 | Completed Pathways For Non-Admitted Patients |
| Part_2 | 63355 | Incomplete Pathways |
| Part_2A | 30548 | Incomplete Pathways with DTA |
| Part_3 | 37354 | New RTT Periods - All Patients |


## treatment_function — 24 distinct code(s), 24 distinct name(s)

Source columns: `Treatment Function Code` / `Treatment Function Name`

Code/name mapping is 1:1 across the raw values in this file.

| code | row_count | name |
| --- | --- | --- |
| C_100 | 9149 | General Surgery Service |
| C_101 | 9047 | Urology Service |
| C_110 | 15507 | Trauma and Orthopaedic Service |
| C_120 | 8673 | Ear Nose and Throat Service |
| C_130 | 13028 | Ophthalmology Service |
| C_140 | 7219 | Oral Surgery Service |
| C_150 | 2218 | Neurosurgical Service |
| C_160 | 4279 | Plastic Surgery Service |
| C_170 | 1138 | Cardiothoracic Surgery Service |
| C_300 | 1596 | General Internal Medicine Service |
| C_301 | 6986 | Gastroenterology Service |
| C_320 | 6294 | Cardiology Service |
| C_330 | 4777 | Dermatology Service |
| C_340 | 4345 | Respiratory Medicine Service |
| C_400 | 3867 | Neurology Service |
| C_410 | 3254 | Rheumatology Service |
| C_430 | 1388 | Elderly Medicine Service |
| C_502 | 9243 | Gynaecology Service |
| C_999 | 40636 | Total |
| X02 | 9736 | Other - Medical Services |
| X03 | 502 | Other - Mental Health Services |
| X04 | 5808 | Other - Paediatric Services |
| X05 | 8716 | Other - Surgical Services |
| X06 | 5005 | Other - Other Services |



## Per-column facts

Straight from `prof.columns` (no pandas dependency). `inferred_dtype` is a
shape hint, **not** a schema; `null` = empty or whitespace-only.

In [5]:
hdr = f"{'#':>3}  {'column':<40} {'dtype':<16} {'nulls':>9} {'null%':>7} {'nuniq':>8}"
print(hdr)
print("-" * len(hdr))
for c in prof.columns:
    print(f"{c.position:>3}  {c.name:<40} {c.inferred_dtype:<16} "
          f"{c.n_null:>9,} {c.null_pct:>7.2f} {c.n_unique_est:>8,}")

  #  column                                   dtype                nulls   null%    nuniq
-----------------------------------------------------------------------------------------
  0  Period                                   string                   0    0.00        1
  1  Provider Parent Org Code                 string                   0    0.00       36
  2  Provider Parent Name                     string                   0    0.00       36
  3  Provider Org Code                        string                   0    0.00      537
  4  Provider Org Name                        string                   0    0.00      536
  5  Commissioner Parent Org Code             string              10,263    5.63       36
  6  Commissioner Parent Name                 string              10,263    5.63       36
  7  Commissioner Org Code                    string                   0    0.00      129
  8  Commissioner Org Name                    string               3,348    1.84      121
  9  RTT P

## Column flags (header text only)

`Total`, `Unknown`, and week-bucket columns identified **by header string
only** — no semantic meaning is assigned here. Whether the week buckets form a
clean sequence is a separate structural check (see above). Full lists in
`outputs/profiles/<stem>__column_flags.csv`.

In [6]:
for flag, cols in prof.column_flags.items():
    preview = cols if len(cols) <= 8 else [cols[0], "...", cols[-1]]
    print(f"{flag:<20} {len(cols):>4}  {preview}")

total_columns           2  ['Total', 'Total All']
unknown_columns         1  ['Patients with unknown clock start date']
week_bucket_columns   105  ['Gt 00 To 01 Weeks SUM 1', '...', 'Gt 104 Weeks SUM 1']


## Categorical dimensions

Distinct codes / names for the semantic roles the profiler recognises
(provider, provider parent, commissioner, RTT part, treatment function,
reporting period).

In [7]:
for role, cat in prof.categoricals.items():
    tail = f" / {cat.name_column!r}" if cat.name_column else ""
    print(f"\n=== {role}  ({cat.n_distinct_codes} distinct codes) "
          f"from {cat.code_column!r}{tail} ===")
    rows = cat.rows()
    for r in rows[:25]:
        print("  ", r)
    if len(rows) > 25:
        print(f"   ... {len(rows) - 25} more — see "
              f"outputs/profiles/{RAW_CSV.stem}__{role}.csv")


=== reporting_period  (1 distinct codes) from 'Period' ===
   {'code': 'RTT-June-2026', 'row_count': 182411}

=== provider  (537 distinct codes) from 'Provider Org Code' / 'Provider Org Name' ===
   {'code': 'A0C5S', 'row_count': 22, 'name': 'SPAMEDICA GLOUCESTER'}
   {'code': 'A1D1B', 'row_count': 54, 'name': 'SPAMEDICA WEMBLEY'}
   {'code': 'A1S9E', 'row_count': 38, 'name': 'OPTEGRA EYE HOSPITAL YORK'}
   {'code': 'A1U4J', 'row_count': 36, 'name': 'SPAMEDICA PORTSMOUTH'}
   {'code': 'A4M8P', 'row_count': 287, 'name': 'BUCKSHAW HOSPITAL'}
   {'code': 'A4Q9X', 'row_count': 60, 'name': 'NEWMEDICA LEICESTER'}
   {'code': 'A4Y8X', 'row_count': 28, 'name': 'ACES NOTTINGHAM'}
   {'code': 'A5E1F', 'row_count': 64, 'name': 'OPTEGRA EYE HOSPITAL PRESTON'}
   {'code': 'A6U7U', 'row_count': 36, 'name': 'OPTEGRA EYE HOSPITAL NOTTINGHAM'}
   {'code': 'A8L1O', 'row_count': 34, 'name': 'OPTEGRA EYE HOSPITAL LEICESTER'}
   {'code': 'A9P5F', 'row_count': 62, 'name': 'ACES WHITE CITY'}
   {'code': 'A9

## Code ↔ name mapping consistency

The brief warns against assuming provider name ↔ code is 1:1. The profiler
reports — but does not resolve — every non-1:1 relationship (including a
blank-vs-blank pair).

In [8]:
any_anom = False
for role, cat in prof.categoricals.items():
    for row in cat.mapping_anomaly_rows():
        any_anom = True
        print(f"[{role}] {row['kind']}: {row['key']!r} -> "
              f"{row['n_related']} values: {row['related_values']}")
if not any_anom:
    print("No code/name mapping anomalies in this file.")

[provider] name_has_multiple_codes: 'DUCHY HOSPITAL' -> 2 values: NT447 | NVC04
[commissioner] name_has_multiple_codes: '' -> 8 values: NONC | Y56 | Y58 | Y59 | Y60 | Y61 | Y62 | Y63


## Observations (structural only)

What the numbers *show*, not what they *mean*. Interpretation of blanks,
subtotal-vs-population rows, and the `Total` vs `Total All` relationship is
deferred to later stages and to `docs/assumptions.md` / `docs/decisions.md`.

- 182,411 rows × 121 columns; 0 exact whole-row duplicates; 0 ragged rows;
  no structural warnings (duplicate headers, broken quoting, week-bucket
  sequence all clean).
- 13 identifier/label columns, then 105 week-bucket columns
  (`Gt 00 To 01 Weeks SUM 1` … `Gt 104 Weeks SUM 1`: 104 contiguous unit-width
  bands + one open-ended), then `Total`,
  `Patients with unknown clock start date`, `Total All`.
- `Total All` is never blank (min 1); `Total` is blank on ~72 % of rows;
  the unknown-clock column is blank on ~81 %; week buckets on ~28–33 %.
  **This does not tell us blank = 0.**
- `Provider Org Code` has 537 distinct values but `Provider Org Name` only
  536: `DUCHY HOSPITAL` is carried by both `NT447` and `NVC04`.
- `Commissioner Org Name` is blank on 3,348 rows; 8 commissioner codes
  (`NONC`, `Y56`, `Y58`–`Y63`) carry an empty name (a blank collision).
- `Treatment Function Code` `C_999` carries the literal name `Total` on
  40,636 rows, alongside 23 other codes. Whether it is a subtotal, a distinct
  population, or something else is **not established** (see assumptions A-07).
- Single `Period` value: `RTT-June-2026`.